In [0]:
# ============================================================
# Silver — Source 03: MongoDB Atlas Products
#
# Transformations:
#   - Cast changed_at nanoseconds to timestamp
#   - Validate price_pence > 0
#   - Deduplicate on product_sku — keep latest changed_at
#   - Normalise collection to lowercase
#
# Source:  bronze.src_03_products.products
# Target:  silver.src_03_products.products
# Quarantine: silver.quarantine.src_03_products
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_03_products.products'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_03_products'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_03_products')
print('Silver Source 03 MongoDB — starting...')


In [0]:
# ── LOAD AND CLEAN ────────────────────────────────────────────
bronze = spark.table(f'{BRONZE_CATALOG}.src_03_products.products')
total = bronze.count()
print(f'Bronze rows: {total}')

# Step 1: Cast nanoseconds to timestamp
df = bronze.withColumn('changed_at', (F.col('changed_at') / 1e9).cast('timestamp'))

# Step 2: Normalise
df = df \
    .withColumn('collection', F.lower(F.trim(F.col('collection')))) \
    .withColumn('reason', F.lower(F.trim(F.col('reason')))) \
    .withColumn('product_sku', F.upper(F.trim(F.col('product_sku'))))

# Step 3: Identify bad rows
bad = df.filter(
    F.col('product_sku').isNull() |
    F.col('price_pence').isNull() |
    (F.col('price_pence') <= 0)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('products'))

# Step 4: Good rows — deduplicate on product_sku, keep latest
good = df.filter(
    F.col('product_sku').isNotNull() &
    F.col('price_pence').isNotNull() &
    (F.col('price_pence') > 0)
)

w = Window.partitionBy('product_sku').orderBy(F.col('changed_at').desc())
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
print(f'Products: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Step 5: Write to Silver
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.product_sku = s.product_sku') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

# Step 6: Quarantine
if bad_count > 0:
    quarantine = bad.select(
        F.lit('src_03_products').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )
    quarantine.write.format('delta').mode('append') \
        .option('mergeSchema', 'true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} rows quarantined')

print(f'\n✅ {TARGET_TABLE}: {good_count} rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
unique_skus = spark.sql(f'SELECT COUNT(DISTINCT product_sku) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_03_products.products: {count} rows, {unique_skus} unique SKUs')
spark.sql(f'SELECT product_sku, price_pence, changed_at, reason FROM {TARGET_TABLE} LIMIT 5').show(truncate=False)
